# Example: degenerate data

**The problem.** Real samples contain values that break the picture of a smooth mixture: a sensor stuck at one
reading, single outliers. A component built on such a group is almost a spike. Here the log-likelihood becomes a
trap: a spike explains its few points extremely well, so a start with a spike has a **higher** likelihood than a
sensible start — and EM never leaves the spike. This notebook shows the two checks that keep such starts out, and
what happens without them.

**How to read the numbers.** The sample has two real modes, `N(0, 1000)` and `N(6000, 1500)`, plus 40 stuck readings.
A good fit has one component near each mode; the stuck readings belong to no real mode.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from pysatl_core.families import ParametricFamilyRegister
from pysatl_core.types import FamilyName

from pysatl_mpest.core import MixtureModel
from pysatl_mpest.estimators import EM
from pysatl_mpest.estimators.iterative import StepBreakpointer
from pysatl_mpest.optimizers import ScipyNelderMead

normal = ParametricFamilyRegister.get(FamilyName.NORMAL)
expon = ParametricFamilyRegister.get(FamilyName.EXPONENTIAL)
uniform = ParametricFamilyRegister.get(FamilyName.CONTINUOUS_UNIFORM)


def show(mixture: MixtureModel) -> None:
    for weight, component in zip(mixture.weights, mixture):
        parameters = {name: round(value, 3) for name, value in component.parameters.items()}
        print(f"weight={weight:.3f}  {component.family_name} {parameters}")


def em(steps: int) -> EM:
    return EM(breakpointers=[StepBreakpointer(steps)], pruners=[], optimizer=ScipyNelderMead())


def plot_mixtures(ax, X, mixtures, bins=60):
    grid = np.linspace(X.min(), X.max(), 1000)
    ax.hist(X, bins=bins, density=True, color="lightgray")
    for label, mixture in mixtures.items():
        ax.plot(grid, np.exp(mixture.lpdf(grid)), label=label)
    ax.legend()

def gap(mixture: MixtureModel, X: np.ndarray, truth: MixtureModel) -> float:
    """How much worse the mixture explains X than the true mixture; near 0 is as good as the truth."""
    return mixture.loglikelihood(X) - truth.loglikelihood(X)


def describe(component) -> str:
    return ", ".join(f"{name}={value:.3g}" for name, value in component.parameters.items())


def compare(truth: MixtureModel, fitted: MixtureModel, labels: list[str]) -> None:
    """Print the true and the fitted components side by side."""
    print(f"{'component':22} {'true':42} {'fitted':42}")
    for label, true, est, w_true, w_est in zip(labels, truth, fitted, truth.weights, fitted.weights):
        left, right = f"share {w_true:.2f}: {describe(true)}", f"share {w_est:.2f}: {describe(est)}"
        print(f"{label:22} {left:42} {right:42}")


def plot_components(ax, X, mixture, title, truth=None, labels=None, bins=60, xlabel="x"):
    """Histogram of X, every component of the mixture separately and the true mixture dashed."""
    grid = np.linspace(X.min(), X.max(), 1000)
    ax.hist(X, bins=bins, density=True, color="lightgray")
    for j, (weight, component) in enumerate(zip(mixture.weights, mixture)):
        label = labels[j] if labels else f"component {j}"
        ax.plot(grid, weight * np.exp(MixtureModel([component]).lpdf(grid)), label=label)
    if truth is not None:
        ax.plot(grid, np.exp(truth.lpdf(grid)), "k--", linewidth=1, label="true mixture")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.legend(fontsize=8)

from pysatl_mpest.initializers import (
    ClusterInitializer,
    InitializationError,
    KMeansPartitioner,
    MultiStartInitializer,
    RandomCentersPartitioner,
)

## A stuck sensor

The data: two wide modes and 40 readings stuck near 15000 (the thin bar on the right).

In [ ]:
rng = np.random.default_rng(42)
X = np.concatenate([rng.normal(0, 1000, 1000), rng.normal(6000, 1500, 1000), 15000 + rng.uniform(-0.001, 0.001, 40)])
modes = MixtureModel([normal(mu=0, sigma=1000), normal(mu=6000, sigma=1500)])
families = [normal, normal]
initializer = ClusterInitializer(RandomCentersPartitioner())

fig, ax = plt.subplots(figsize=(9, 3.5))
plot_components(ax, X, modes, "Two real modes and the stuck readings", labels=["mode at 0", "mode at 6000"], bins=100)
plt.show()

Ten draws of random centers. Some of them put the stuck readings into a cluster of their own. The component on
it would be narrower than a millionth of the spread of the sample, so the **width check** rejects it, and the
multistart skips the run with a warning:

In [ ]:
for child in np.random.default_rng(13).spawn(10):
    try:
        initializer.initialize(X, families, random_state=child)
    except InitializationError as error:
        print("rejected run:", error)
print()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    kept = MultiStartInitializer(initializer, n_init=10).initialize(X, families, random_state=13)
print("warning of the multistart:", caught[0].message)

**What the rejected run would give.** Build its start by hand — the stuck readings as one component, the rest
as the other — and compare it with the kept start. The spike start has the higher likelihood: without the check it
would win the multistart.

In [ ]:
ordinary, stuck = X[:2000], X[2000:]
spike = MixtureModel(
    [normal(mu=ordinary.mean(), sigma=ordinary.std()), normal(mu=stuck.mean(), sigma=stuck.std())],
    weights=[2000 / 2040, 40 / 2040],
)
print(f"log-likelihood: spike start {spike.loglikelihood(X):.0f}, kept start {kept.loglikelihood(X):.0f}")
from_spike, from_kept = em(30).fit(X, spike), em(30).fit(X, kept)

fig, axes = plt.subplots(1, 2, figsize=(16, 3.5), sharey=True)
plot_components(axes[0], X, from_spike, "EM from the spike start", truth=modes, bins=100)
plot_components(axes[1], X, from_kept, "EM from the kept start", truth=modes, bins=100)
plt.show()
print("EM from the spike start:")
compare(modes, from_spike, ["mode at 0", "mode at 6000"])
print("\nEM from the kept start:")
compare(modes, from_kept, ["mode at 0", "mode at 6000"])

assert spike.loglikelihood(X) > kept.loglikelihood(X)
assert abs(from_kept[1].parameters["mu"] - 6000) < 500

**Reading the result.** From the spike start, one component stays on the stuck readings and the other covers
both real modes with one wide bell (dashed: what it should be). From the kept start, EM finds both modes.

## An outlier

A small sample of 20 points with one value at 100. Without the size threshold (`min_cluster_size=1`) k-means gives
the outlier a cluster of its own, and the start has a component on one point. With the default threshold — at least
4 points per cluster for `Normal`, twice its two free parameters — such a partition is rejected, and a multistart
looks for another one.

In [ ]:
rng = np.random.default_rng(42)
small = np.concatenate([rng.normal(0, 1, 20), [100.0]])

lax = ClusterInitializer(KMeansPartitioner(), min_cluster_size=1).initialize(small, [normal, expon], random_state=0)
print("min_cluster_size=1 — the second component is built on the outlier alone:")
for weight, component in zip(lax.weights, lax):
    print(f"  share {weight:.3f}: {component.family_name} {describe(component)}")

try:
    ClusterInitializer(KMeansPartitioner()).initialize(small, [normal, normal], random_state=0)
except InitializationError as error:
    print("\ndefault threshold:", error)

## Takeaway

Both checks are on by default and need no tuning: the width check catches groups of almost equal values, the size
threshold catches clusters of a few points. Neither silently replaces a bad component: the run fails, and a
multistart finds another one.